### 임베딩
- OpenAIEmbeddings

In [ ]:
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH08-Embeddings")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
text = "임베딩 테스트를 하기 위한 샘플 문장입니다."

In [ ]:
query_result = embeddings.embed_query(text) # 텍스트를 임베딩

In [ ]:
len(query_result) # 임베딩 결과의 길이 확인

1536

In [ ]:
query_result[:5] # 임베딩 된 결과의 쿼리로 처음 5개 항목을 선택

[-0.007762908935546875,
 0.036712646484375,
 0.01953125,
 -0.0196990966796875,
 0.0172119140625]

In [ ]:
doc_result = embeddings.embed_documents(
    [text, text, text, text]
) # 텍스트를 임베딩하여 문서 벡터를 생성

In [ ]:
doc_result[0][:5] # 문서 벡터의 첫 번째 항목에서 처음 5개 항목을 선택

[-0.00775146484375,
 0.036712646484375,
 0.01953125,
 -0.0196990966796875,
 0.0172119140625]

In [ ]:
len(doc_result[0]) # 문서 벡터의 길이 확인

1536

In [ ]:
embeddings_1024 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

len(embeddings_1024.embed_documents([text])[0]) # 1024차원 임베딩 결과의 길이 확인

1024

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = "안녕하세요? 반갑습니다."
sentence2 = "안녕하세요? 반갑습니다!"
sentence3 = "안녕하세요? 만나서 반가워요."
sentence4 = "Hi, nice to meet you."
sentence5 = "I like to eat apples."

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]
embedded_sentences = embeddings_1024.embed_documents(sentences)

In [ ]:
def similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

In [ ]:
for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[유사도 {similarity(sentence, other_sentence):.4f}] {sentences[i]} \t <=====> \t {sentences[j]}"
            )

[유사도 0.9644] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 반갑습니다!
[유사도 0.8422] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.5043] 안녕하세요? 반갑습니다. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1362] 안녕하세요? 반갑습니다. 	 <=====> 	 I like to eat apples.
[유사도 0.8185] 안녕하세요? 반갑습니다! 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.4791] 안녕하세요? 반갑습니다! 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1319] 안녕하세요? 반갑습니다! 	 <=====> 	 I like to eat apples.
[유사도 0.5164] 안녕하세요? 만나서 반가워요. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1459] 안녕하세요? 만나서 반가워요. 	 <=====> 	 I like to eat apples.
[유사도 0.2250] Hi, nice to meet you. 	 <=====> 	 I like to eat apples.


### 영구적으로 임베딩을 저장하는 LocalFileStore

In [3]:
from langchain_classic.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH08-Embeddings")

embedding = OpenAIEmbeddings() # 기본 임베딩 설정

store = LocalFileStore("./cache/") # 로컬 파일 스토어 생성

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


C:\Users\user\AppData\Local\Temp\ipykernel_13944\1961666065.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding, # 임베딩 모델 지정
    document_embedding_cache=store, # 임베딩 캐시 지정 ./cache/ 폴더
    namespace=embedding.model, # 임베딩 값 구분자
)

d:\psj0902\ex0922\ex0922\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [ ]:
list(store.yield_keys()) # 캐시된 임베딩 키 확인

[]

In [5]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader(
    "./data/appendix-keywords.txt", encoding="utf-8").load() # 텍스트 파일 로드
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0) # 텍스트 분할기 생성
documents = text_splitter.split_documents(raw_documents) # 텍스트 분할

In [6]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 31.2 ms
Wall time: 42.2 ms


In [7]:
%time db2 = FAISS.from_documents(documents, cached_embedder) # 캐시된 임베딩을 사용하여 FAISS 벡터 스토어 생성

CPU times: total: 31.2 ms
Wall time: 12.4 ms


### 비영구적으로 임베딩을 저장하는 InMemoryByteStore

In [12]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import InMemoryByteStore

store = InMemoryByteStore() # 임베딩 캐시를 위한 메모리 스토어 생성 메모리 내 바이트 저장소 생성

#캐시 지원 임베딩 생성
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)